# Generating input for MODFLOW


## FloPy
- Bakker, M., Post, V., Langevin, CD., Hughes, JD., White, J. T., Starn, J. J., & Fienen, M. N. (2016). Scripting MODFLOW Model Development Using Python and FloPy. Groundwater, 54(5), 733-739. https://doi.org/10.1111/gwat.12413/epdf
- Hughes, J.D., Langevin, C.D., Paulinski, S.R., Larsen, J.D. and Brakenhoff, D. (2024), FloPy Workflows for Creating Structured and Unstructured MODFLOW Models. Groundwater, 62: 124-139. https://doi.org/10.1111/gwat.13327

In [1]:
import flopy as fp
import numpy as np
import pandas as pd

import pedon as pe

## Converting soil models

In [3]:
pe.get_params(pe.Panday)

,p_ini,p_min,p_max
k_s,50.00,0.00100,100000.0
theta_r,0.02,0.00001,0.2
theta_s,0.40,0.20000,0.5
alpha,0.02,0.00100,0.3
beta,2.30,1.00000,12.0
brook,10.00,1.00000,50.0


In [7]:
gen = pe.Soil("Sand").from_name(pe.Genuchten, source="HYDRUS").model
# gen = pe.Genuchten(k_s=100.0, theta_s=0.4, theta_r=0.05, alpha=0.1, n=2.0)

h = np.logspace(-4, 6, num=11)
theta = gen.theta(h)
k = gen.k(h)

# set bounds for optimization
pbounds = pd.DataFrame(
    data=[
        [getattr(gen, "k_s"), getattr(gen, "k_s") - 1e-10, getattr(gen, "k_s") + 1e-10],
        [
            getattr(gen, "theta_r"),
            getattr(gen, "theta_r") - 1e-10,
            getattr(gen, "theta_r") + 1e-10,
        ],
        [
            getattr(gen, "theta_s"),
            getattr(gen, "theta_s") - 1e-10,
            getattr(gen, "theta_s") + 1e-10,
        ],
        [
            getattr(gen, "alpha"),
            getattr(gen, "alpha") - 1e-10,
            getattr(gen, "alpha") + 1e-10,
        ],
        [getattr(gen, "n"), getattr(gen, "n") - 1e-10, getattr(gen, "n") + 1e-10],
        [10.0, 1.0, 50.0],
    ],
    index=["k_s", "theta_r", "theta_s", "alpha", "beta", "brook"],
    columns=["p_ini", "p_min", "p_max"],
)

pan = pe.SoilSample(h=h, theta=theta, k=k).fit(pe.Panday, pbounds=pbounds, k_s=gen.k_s)
pan

Panday(k_s=712.8, alpha=np.float64(0.14500000009999997), beta=np.float64(2.679999999904514), brook=np.float64(3.7548440249344366), sr=np.float64(0.10465116278938343))

## MODFLOW-USG Tranport
Soil Model
    hk=1.0
    sy=0.15
    alpha=1.0,
    beta=7.0,
    sr=0.05,
    brook=6.0,
    bp=0.0,


In [8]:
fp.mfusg.MfUsgLpf(
    ...,
    hk=pan.k_s / 100.0,
    sy=pan.theta_s,
    ss=pan.ss,
    alpha=pan.alpha * 100,
    beta=pan.beta,
    sr=pan.sr,
    brook=pan.brook,
)

AssertionError: Model object must be of type flopy.mfusg.MfUsg
but received type: <class 'ellipsis'>.

## MODFLOW UZF
No soil water retention curve

In [ ]:
# packagedata : [(ifno, cellid, landflag, ivertcon, surfdep, vks, thtr, thts, thti, eps, boundname)]
packagedata = [
    (..., ..., ..., ..., ..., pan.k_s / 100.0, pan.theta_r, pan.theta_s, ..., pan.brook)
]
fp.mf6.ModflowGwfuzf(
    ...,
    packagedata=packagedata,
)

Init signature:
fp.mf6.ModflowGwfuzf(
    model,
    loading_package=False,
    auxiliary=None,
    auxmultname=None,
    boundnames=None,
    print_input=None,
    print_flows=None,
    save_flows=None,
    wc_filerecord=None,
    budget_filerecord=None,
    budgetcsv_filerecord=None,
    package_convergence_filerecord=None,
    timeseries=None,
    observations=None,
    mover=None,
    simulate_et=None,
    linear_gwet=None,
    square_gwet=None,
    simulate_gwseep=None,
    unsat_etwc=None,
    unsat_etae=None,
    nuzfcells=None,
    ntrailwaves=7,
    nwavesets=40,
    packagedata=None,
    perioddata=None,
    filename=None,
    pname=None,
    **kwargs,
)
Docstring:     
ModflowGwfuzf defines a UZF package.

Parameters
----------
model
    Model that this package is a part of. Package is automatically
    added to model when it is initialized.
loading_package : bool, default False
    Do not set this parameter. It is intended for debugging and internal
    processing purposes 